# Week 1 Assignment · From Tokens to Your First AI Inspector
### Build Custom AI — SarasAI · *ungraded practice*

The live session went from Transformer foundations to a measured VLM defect detector.
This assignment walks the **same arc with your hands on the keyboard** — 12 tasks in five parts:

| Part | Tasks | What you practice |
|---|---|---|
| 1 · Foundations | 1–3 | tokenizers, embedding geometry, `pipeline()` |
| 2 · Meet the VLM | 4–6 | loading a VLM, the chat-message API, robust output parsing |
| 3 · The prompting ladder | 7–9 | zero-shot → domain prompt → **few-shot with images** |
| 4 · Structured output | 10 | JSON contract + defensive parsing |
| 5 · Measure like a pro | 11–12 | frozen dev/test split, precision & recall, confusion matrix |

Every task has `...` placeholders and most are followed by a **self-check cell** — run it to know
immediately whether your code is right. Stuck ≥15 min? Peek at *that one task* in the solution
notebook, close it, and write your own version. ⏱ ~60–90 min on a free Colab T4.

> 📘 **This is the SOLUTION notebook.** One reference implementation — yours may differ and
> be equally correct. Prompts especially: if yours scores better on the same images, keep yours.

---
## Setup (given — run and move on)

In [ ]:
!pip install -q "transformers>=4.50" datasets accelerate pillow scikit-learn matplotlib

In [ ]:
import torch

# T4s (pre-Ampere) have no native bfloat16 — fall back to float16
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}   dtype: {DTYPE}")

---
## Part 1 · Foundations — three ideas, three tasks

Before touching images, prove to yourself the three claims from the session: text becomes
**tokens**, tokens become **vectors whose geometry is meaning**, and the HF `pipeline()` wraps
all of it in one line.

### ✏️ Task 1 — tokenizers — count what the model actually sees

> 💡 **Hint:** tok.tokenize() shows the sub-word pieces; tok.encode() gives the integer IDs the model sees.

In [ ]:
# ─── Task 1 · SOLUTION ───
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

sentences = [
    "The bottle cap is cracked.",
    "Supercalifragilisticexpialidocious!",
    "AI",
]

for s in sentences:
    pieces = tok.tokenize(s)
    ids = tok.encode(s)
    print(f"{s[:40]:42} words:{len(s.split()):2}  tokens:{len(ids):2}  pieces: {pieces[:6]}")

In [ ]:
# ── self-check: token count ≠ word count ──
assert len(tok.encode("Supercalifragilisticexpialidocious!")) > 3, \
    "a rare word should split into MANY sub-word pieces"
print("✅ rare words shatter into pieces — you pay compute per token, not per word")

### ✏️ Task 2 — embedding geometry — measure that meaning is distance

> 💡 **Hint:** F.cosine_similarity(vec_a, vec_b, dim=0) — both vectors come from word_vec().

In [ ]:
# ─── Task 2 · SOLUTION ───
from transformers import AutoModel
import torch.nn.functional as F

lm = AutoModel.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct", torch_dtype=torch.float32)
emb = lm.get_input_embeddings()

def word_vec(word):
    """Average the embedding vectors of a word's tokens."""
    ids = tok.encode(word, add_special_tokens=False)
    return emb(torch.tensor(ids)).mean(dim=0)

sim_defects = F.cosine_similarity(word_vec("crack"), word_vec("fracture"), dim=0)
sim_random  = F.cosine_similarity(word_vec("crack"), word_vec("banana"), dim=0)

print(f"cos(crack, fracture) = {sim_defects:.3f}")
print(f"cos(crack, banana)   = {sim_random:.3f}")

In [ ]:
# ── self-check: related words must be measurably closer ──
assert sim_defects > sim_random, "crack should be closer to fracture than to banana!"
print("✅ meaning is geometry — Week 2 turns this exact fact into a search engine")

### ✏️ Task 3 — pipeline() — the one-liner that wraps all of it

> 💡 **Hint:** pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english") — pin the model so results are stable. Calling it returns a list with one dict per input.

In [ ]:
# ─── Task 3 · SOLUTION ───
from transformers import pipeline

clf = pipeline("sentiment-analysis",
               model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
reviews = ["The kettle works perfectly, love it!",
           "Lid cracked after two weeks. Disappointed."]
for r in reviews:
    result = clf(r)[0]
    print(f"{result['label']:9} {result['score']:.2f}  {r}")

In [ ]:
# ── self-check ──
neg = clf("Lid cracked after two weeks. Disappointed.")[0]
assert neg["label"].upper().startswith("NEG"), "the cracked-lid review should be NEGATIVE"
print("✅ pipeline() = model + tokenizer + pre/post-processing in one object")

---
## Part 2 · Meet the VLM

Now the real thing. The dataset adapter is given (it's the session's fixed version — good
images live in MVTec's *train* split, defects in *test*, masks excluded by path). **20 images
this time**: 10 good, 10 defective — enough to split into dev and test later.

In [ ]:
from huggingface_hub import snapshot_download
from PIL import Image
import glob

# fetch ONLY the bottle folders (~150 MB) — never "the whole dataset" (~5 GB)
DATA_ROOT = snapshot_download("TheoM55/mvtec_anomaly_detection", repo_type="dataset",
                              allow_patterns=["images/train/bottle/*", "images/test/bottle/*"])

def load_bottles(per_class=10):
    """MVTec convention: good units live in train, defects in test."""
    good = sorted(glob.glob(f"{DATA_ROOT}/images/train/bottle/good/*.png"))[:per_class]
    bad = sorted(p for p in glob.glob(f"{DATA_ROOT}/images/test/bottle/*/*.png")
                 if "/good/" not in p)[:per_class]
    return ([(Image.open(p).convert("RGB"), "good") for p in good] +
            [(Image.open(p).convert("RGB"), "defective") for p in bad])

items = load_bottles()
print(f"{len(items)} images:", {l: sum(1 for _, x in items if x == l) for l in ('good', 'defective')})

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
show = items[:5] + items[10:15]            # 5 good, 5 defective
for ax, (img, lbl) in zip(axes.flat, show):
    ax.imshow(img); ax.set_title(lbl, fontsize=10); ax.axis("off")
plt.tight_layout()

### ✏️ Task 4 — load the VLM + processor

> 💡 **Hint:** AutoProcessor.from_pretrained + AutoModelForImageTextToText.from_pretrained — VLMs use a processor (tokenizer + image preprocessor), not a bare tokenizer.

In [ ]:
# ─── Task 4 · SOLUTION ───
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = ("HuggingFaceTB/SmolVLM2-2.2B-Instruct" if device == "cuda"
            else "HuggingFaceTB/SmolVLM-256M-Instruct")

processor = AutoProcessor.from_pretrained(MODEL_ID)
vlm = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map="auto")
print(f"loaded {MODEL_ID}")

### ✏️ Task 5 — implement `ask_vlm(image, question)`

> 💡 **Hint:** the image block carries the PIL image object itself; the text block carries the question string.

In [ ]:
# ─── Task 5 · SOLUTION ───
def ask_vlm(image, question, max_new_tokens=100):
    """Send one image + one question to the VLM, return its text answer."""
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt",
    ).to(vlm.device, dtype=DTYPE)

    out = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = processor.decode(out[0], skip_special_tokens=True)
    return text.split("Assistant:")[-1].strip()

In [ ]:
# ── self-check ──
answer = ask_vlm(items[0][0], "Describe this image in one sentence.")
assert isinstance(answer, str) and len(answer) > 10, "ask_vlm should return a non-trivial string"
print("✅ ask_vlm works\n")
print(answer)

### ✏️ Task 6 — the parser — harder than it looks

> 💡 **Hint:** check the negation phrases FIRST, then fall back to the substring test.

In [ ]:
# ─── Task 6 · SOLUTION ───
def parse_verdict(answer):
    """Map free text onto 'good' / 'defective', handling negations."""
    a = answer.upper()
    for neg in ["NO DEFECT", "NOT DEFECTIVE", "WITHOUT DEFECT", "NO VISIBLE DEFECT"]:
        if neg in a:
            return "good"
    return "defective" if "DEFECT" in a else "good"

for text in ["DEFECTIVE", "The bottle is GOOD.", "No defects found.",
             "This bottle is not defective.", "There is a crack, clearly defective."]:
    print(f"{parse_verdict(text):10}  <-  {text}")

In [ ]:
# ── self-check: the negation trap ──
assert parse_verdict("No defects found.") == "good"
assert parse_verdict("This bottle is not defective.") == "good"
assert parse_verdict("A crack — DEFECTIVE.") == "defective"
print("✅ parser survives negations — this bug ships to production ALL the time")

---
## Part 3 · The prompting ladder

Rule for this whole part: **iterate on the DEV set only.** The split is given — 10 dev images,
10 test images, fixed seed. The test set stays untouched until Task 12.

In [ ]:
import random

rng = random.Random(42)
shuffled = items[:]
rng.shuffle(shuffled)
dev_set, test_set = shuffled[:10], shuffled[10:]
print("dev :", [l for _, l in dev_set])
print("test:", [l for _, l in test_set], " 🔒 frozen until Task 12")

### ✏️ Task 7 — the accuracy loop + zero-shot baseline

> 💡 **Hint:** ask with max_new_tokens=10 (one-word answers), parse, compare to truth, count.

In [ ]:
# ─── Task 7 · SOLUTION ───
def accuracy(dataset, prompt):
    """Fraction of images where parse_verdict(ask_vlm(...)) matches the true label."""
    correct = 0
    for img, truth in dataset:
        pred = parse_verdict(ask_vlm(img, prompt, max_new_tokens=10))
        correct += (pred == truth)
    return correct / len(dataset)

PROMPT_V1 = "Is this product GOOD or DEFECTIVE? Answer with exactly one word."

acc_v1 = accuracy(dev_set, PROMPT_V1)
print(f"v1 zero-shot (dev): {acc_v1:.0%}")

### ✏️ Task 8 — the domain prompt

> 💡 **Hint:** role + defect checklist + anti-false-positive rule + output format. Four sentences is plenty.

In [ ]:
# ─── Task 8 · SOLUTION ───
PROMPT_V2 = (
    "You are a quality-control inspector for glass bottles. "
    "Look carefully for: cracks or fractures, chipped or broken glass at the rim or neck, "
    "and contamination inside the bottle. "
    "Lighting reflections and glare are NOT defects. "
    "Is this bottle GOOD or DEFECTIVE? Answer with exactly one word."
)

acc_v2 = accuracy(dev_set, PROMPT_V2)
print(f"dev — v1: {acc_v1:.0%}   v2: {acc_v2:.0%}")

### ✏️ Task 9 — few-shot — show it labeled example IMAGES

> 💡 **Hint:** each exemplar = an image block followed by a text block with its label. Exemplars come from dev because test must never influence ANY design choice — including which examples you show.

In [ ]:
# ─── Task 9 · SOLUTION ───
def predict_fewshot(image, examples, question):
    """Build ONE user message whose content interleaves labeled example images, then the query."""
    content = [{"type": "text", "text":
        "You are a quality-control inspector for glass bottles. Labeled examples:"}]
    for ex_img, ex_label in examples:
        content += [{"type": "image", "image": ex_img},
                    {"type": "text", "text": f"Label: {ex_label.upper()}"}]
    content += [{"type": "text", "text": question}, {"type": "image", "image": image}]

    inputs = processor.apply_chat_template([{"role": "user", "content": content}],
        add_generation_prompt=True, tokenize=True, return_dict=True,
        return_tensors="pt").to(vlm.device, dtype=DTYPE)
    out = vlm.generate(**inputs, max_new_tokens=10, do_sample=False)
    return parse_verdict(processor.decode(out[0], skip_special_tokens=True).split("Assistant:")[-1])

shots = [next(x for x in dev_set if x[1] == "good"),
         next(x for x in dev_set if x[1] == "defective")]
QUESTION = "Now classify this image. Answer GOOD or DEFECTIVE."

acc_fs = sum(predict_fewshot(img, shots, QUESTION) == truth for img, truth in dev_set) / len(dev_set)
print(f"dev — v2: {acc_v2:.0%}   few-shot: {acc_fs:.0%}")

Few-shot may win or lose against v2 here — small VLMs have limited in-context capacity for
images, and each exemplar costs ~1,500 tokens per call. **Accuracy vs. per-call cost is exactly
the trade-off your dev set exists to settle.**

---
## Part 4 · Structured output

### ✏️ Task 10 — JSON verdict + defensive parser

> 💡 **Hint:** re.search(r'\{.*\}', answer, re.DOTALL); always return the same three keys.

In [ ]:
# ─── Task 10 · SOLUTION ───
import json, re

PROMPT_JSON = (
    "You are a quality-control inspector for glass bottles. "
    "Inspect the image and respond with ONLY a JSON object, no other text: "
    '{"verdict": "GOOD" or "DEFECTIVE", "defect_type": short string or null, '
    '"confidence": "low", "medium" or "high"}'
)

def inspect(image):
    """Return a dict with keys verdict / defect_type / confidence — even if the model misbehaves."""
    answer = ask_vlm(image, PROMPT_JSON, max_new_tokens=80)
    m = re.search(r"\{.*\}", answer, re.DOTALL)
    try:
        parsed = json.loads(m.group()) if m else {}
        return {"verdict": str(parsed.get("verdict", "UNKNOWN")).upper(),
                "defect_type": parsed.get("defect_type"),
                "confidence": parsed.get("confidence", "low")}
    except Exception:
        return {"verdict": "UNKNOWN", "defect_type": None, "confidence": "low"}

print(inspect(dev_set[0][0]))

In [ ]:
# ── self-check: inspect() must never crash and always honor the contract ──
for img, _ in dev_set[:3]:
    r = inspect(img)
    assert set(r) == {"verdict", "defect_type", "confidence"}, f"wrong keys: {set(r)}"
print("✅ inspect() honors the contract — this is the capstone's Stage-1 interface")

---
## Part 5 · Measure like a professional

Decision time: pick your best prompt **by dev accuracy**, then touch the test set **once**.

### ✏️ Task 11 — commit, then the single test run — timed

> 💡 **Hint:** if few-shot won on dev, adapt the loop to call predict_fewshot instead — the discipline (one pass, timed) is what matters.

In [ ]:
# ─── Task 11 · SOLUTION ───
import time

FINAL_PROMPT = PROMPT_V2 if acc_v2 >= max(acc_v1, acc_fs) else PROMPT_V1

t0 = time.perf_counter()
y_true = [truth for _, truth in test_set]
y_pred = [parse_verdict(ask_vlm(img, FINAL_PROMPT, max_new_tokens=10)) for img, _ in test_set]
elapsed = time.perf_counter() - t0

GPU_PRICE_PER_H = 0.40
sec_per_img = elapsed / len(test_set)
print(f"test accuracy: {sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true):.0%}")
print(f"{sec_per_img:.2f} s/image  →  ${GPU_PRICE_PER_H / 3600 * sec_per_img * 1000:.2f} per 1k images")

### ✏️ Task 12 — precision, recall & the confusion matrix — then read them

> 💡 **Hint:** classification_report(y_true, y_pred, labels=labels); confusion_matrix with the same label order.

In [ ]:
# ─── Task 12 · SOLUTION ───
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

labels = ["good", "defective"]

print(classification_report(y_true, y_pred, labels=labels, digits=2))

cm = confusion_matrix(y_true, y_pred, labels=labels)
ConfusionMatrixDisplay(cm, display_labels=labels).plot(cmap="Blues")

In [ ]:
# ── final reflection (edit this cell) ──
# 1. Which cell of the confusion matrix is MOST expensive for a factory: a missed defect
#    (false 'good') or a false alarm (false 'defective')? Which metric tracks it?
# 2. Your dev accuracy vs test accuracy — did the ranking of prompts hold? With only 10 test
#    images, how big is the noise on one flipped prediction? (Hint: 10 points!)
ANSWER_1 = "..."
ANSWER_2 = "..."
print("Done. The graded increment scales exactly this discipline to a bigger frozen split.")

---
## Wrap-up · What you practiced

| Task | Skill | Where it goes next |
|---|---|---|
| 1–3 | tokens, embedding geometry, pipeline() | Week 2's retrieval engine |
| 4–6 | VLM API + robust parsing | capstone Stage 1 |
| 7–9 | prompt ladder incl. few-shot, dev-only iteration | every week of this course |
| 10 | JSON contract | the pipeline interface |
| 11–12 | frozen test discipline, P/R, confusion matrix, $/1k | the graded increment's rubric |

*Ungraded — nothing to submit. If all self-checks pass and your reflection answers are honest,
you're ready for graded increment 1.*